In [ ]:
!pip -q install transformers torch torchvision pillow matplotlib

In [ ]:
import torch
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import files
from transformers import CLIPProcessor, CLIPModel

In [ ]:
if torch.cuda.is_available():
    print("GPU is available")
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is NOT available (CPU mode)")

In [ ]:
uploaded = files.upload()

image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

print("image size:", image.size)
image

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)

print("CLIP loaded.")

In [ ]:
candidate_labels = [
    "a laptop",
    "a tablet",
    "a desktop computer",
    "a phone",
    "a projector screen",
    "a classroom"
]

In [ ]:
inputs = processor(
    text=candidate_labels,
    images=image,
    return_tensors="pt",
    padding=True
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1).cpu().numpy()[0]

In [ ]:
print("CLIP classification results:\n")

for label, prob in zip(candidate_labels, probs):
    print(f"{label}: {prob:.4f}")

best_idx = probs.argmax()
best_label = candidate_labels[best_idx]
best_score = probs[best_idx]

print("\nBest prediction:")
print(f"{best_label} (score={best_score:.4f})")

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title(f"CLIP Prediction:\n{best_label} ({best_score:.2f})")
plt.axis("off")
plt.show()